##**Projeto:** Merca Data Platform 

##**Squad:** 2 | Streaming em Tempo Real
### Objetivo
Verificar se o ambiente está pronto antes de iniciar o pipeline.
Este notebook deve ser executado **uma vez** no início de cada sessão ou deploy.
### O que este notebook faz
| Etapa | Descrição |
| 1 | Conecta ao ADLS Gen2 e valida acesso ao container |
| 2 | Lista e valida os snapshots disponíveis no lake |
| 3 | Verifica se as tabelas do Squad 2 existem e podem ser lidas |
| 4 | Cria a estrutura de pastas medalhão no ADLS (se ainda não existir) |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | Funções de conexão, snapshot e log |
### Containers e Caminhos
| Item | Valor |
| Container de leitura | `real-time-ecommerce-data` (variável `ADLS_CONTAINER`) |
| Container de escrita | `squad2` (variável `SQUAD2_CONTAINER`) |
| Estrutura criada | `squad2/bronze/`, `squad2/silver/`, `squad2/gold/` |




In [0]:
%run ../utils/feat_squad2_99_helpers

- Conexão com o ADLS e Listagem do Container
Testa a conexão com o Azure Data Lake e exibe o conteúdo do container raw, diferenciando pastas de arquivos.


In [0]:
inicio = log_inicio("feat_squad2_00_setup_config")

try:
    container_client = get_container_client()
    log.info(f"Conexão com ADLS estabelecida!")
    log.info(f"Container: {ADLS_CONTAINER}")
except Exception as e:
    log.error(f"Erro ao conectar no ADLS: {str(e)}")
    raise

## 3. Listar Estrutura do Container

try:
    itens = list(container_client.get_paths())
    log.info(f"{len(itens)} item(ns) encontrado(s) no container\n")

    for item in itens:
        tipo = "Pastas" if item.is_directory else "Arquivos"
        print(f"  {tipo} {item.name}")

except Exception as e:
    log.error(f"Erro ao listar container: {str(e)}")
    raise

## 4. Validar Snapshots Disponíveis

try:
    snapshots = listar_snapshots()
    log.info(f"{len(snapshots)} snapshot(s) encontrado(s)\n")

    for snap in sorted(snapshots):
        print(f"Pacotes {snap}")

    # Snapshot mais recente
    mais_recente = get_snapshot_mais_recente()
    log.info(f"Snapshot mais recente: {mais_recente}")

except Exception as e:
    log.error(f"Erro ao listar snapshots: {str(e)}")
    raise

## 5. Validar Tabelas do Squad 2

try:
    snap_ref = get_snapshot_mais_recente()
    log.info(f"Validando tabelas no snapshot: {snap_ref}\n")

    for tabela in TABELAS_SQUAD2:
        try:
            df = ler_parquet(snap_ref, tabela)
            log.info(
                f"OK {tabela} → "
                f"{df.count()} linhas | "
                f"{len(df.columns)} colunas"
            )
        except Exception as e:
            log.error(f"Erro em {tabela}: {str(e)}")

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

log_fim("feat_squad2_00_setup_config", inicio)

- Criação da Estrutura Medalhão no ADLS
Garante que as pastas `bronze/`, `silver/` e `gold/` existam no container `squad2`para cada tabela do Squad 2. A pasta `gold/` recebe apenas a pasta `kpis/`.
Um arquivo sentinela `.keep` é criado em cada pasta para forçar a existência do diretório.
Se a pasta já existir, nenhuma ação é tomada.

In [0]:
# Executa apenas se as pastas não existirem
import logging
logging.getLogger("azure").setLevel(logging.WARNING)

ESTRUTURA_MEDALHAO = {
    "bronze": TABELAS_SQUAD2,
    "silver": TABELAS_SQUAD2,
    "gold"  : ["kpis"]
}

container_client = get_container_client()
criados          = []
ja_existia       = []
erros            = []

for camada, tabelas in ESTRUTURA_MEDALHAO.items():
    for tabela in tabelas:
        path        = f"squad2/{camada}/{tabela}/.keep"
        file_client = container_client.get_file_client(path)

        try:
            # Verifica se já existe
            file_client.get_file_properties()
            ja_existia.append(f"squad2/{camada}/{tabela}/")
            log.info(f" Já existe: squad2/{camada}/{tabela}/")

        except Exception:
            # Não existe — cria
            try:
                file_client.create_file()
                criados.append(f"squad2/{camada}/{tabela}/")
                log.info(f" Criado: squad2/{camada}/{tabela}/")
            except Exception as e:
                erros.append(f"squad2/{camada}/{tabela}/")
                log.error(f" Erro: squad2/{camada}/{tabela}/ → {str(e)}")

print(f"\n{'='*50}")
print(f"  ESTRUTURA MEDALHÃO — SQUAD 2")
print(f"{'='*50}")
print(f"   Criados    : {len(criados)}")
print(f"   Já existiam: {len(ja_existia)}")
print(f"   Erros      : {len(erros)}")
print(f"{'='*50}")

if criados:
    log.info("Estrutura criada com sucesso!")
elif ja_existia and not erros:
    log.info("Estrutura já existia — nenhuma ação necessária.")